# 03 - Inference
Load the packaged pipeline and score raw records; then call the same model through the REST API.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 140)

In [2]:
import json
from heart_risk.predict import load_model, load_metadata, predict_records
model = load_model()
meta = load_metadata()
print(meta['algorithm'], meta['test_metrics'])
records = json.loads((ROOT/'scripts/sample_input.json').read_text())
pd.DataFrame(predict_records(model, records))

logistic_regression {'accuracy': 0.8688524590163934, 'precision': 0.8125, 'recall': 0.9285714285714286, 'f1': 0.8666666666666667, 'roc_auc': 0.9610389610389611}


,prediction,label,confidence,risk_probability,risk_band
0,0,no_heart_disease,0.7020,0.2980,low
1,1,heart_disease,0.9965,0.9965,high
2,0,no_heart_disease,0.9343,0.0657,low


Missing optional fields (`ca`, `thal`) are imputed by the pipeline:

In [3]:
partial = {k: v for k, v in records[1].items() if k not in ('ca','thal')}
predict_records(model, [partial])

[{'prediction': 1,
  'label': 'heart_disease',
  'confidence': 0.8771,
  'risk_probability': 0.8771,
  'risk_band': 'high'}]

## Through the API (in-process test client; use `requests` against http://localhost:8000 for the container)

In [4]:
from fastapi.testclient import TestClient
from api.main import create_app
client = TestClient(create_app())
r = client.post('/predict', json=records[0])
r.status_code, r.json()

{"ts": "2026-10-19T02:00:05", "level": "INFO", "logger": "heart_api", "msg": "model loaded", "model_path": "/Users/apmuraleedha/Downloads/heart-risk-mlops/models/model.joblib", "algorithm": "logistic_regression"}


{"ts": "2026-10-19T02:00:05", "level": "INFO", "logger": "heart_api", "msg": "model loaded", "model_path": "/Users/apmuraleedha/Downloads/heart-risk-mlops/models/model.joblib", "algorithm": "logistic_regression"}


{"ts": "2026-10-19T02:00:05", "level": "INFO", "logger": "heart_api", "msg": "prediction", "request_id": "9d4df3a25a06", "input": {"age": 63.0, "sex": 1, "cp": 1, "trestbps": 145.0, "chol": 233.0, "fbs": 1, "restecg": 2, "thalach": 150.0, "exang": 0, "oldpeak": 2.3, "slope": 3, "ca": 0, "thal": 6}, "prediction": 0, "label": "no_heart_disease", "confidence": 0.702, "risk_probability": 0.298, "risk_band": "low"}


{"ts": "2026-10-19T02:00:05", "level": "INFO", "logger": "heart_api", "msg": "request", "request_id": "9d4df3a25a06", "method": "POST", "path": "/predict", "status": 200, "latency_ms": 8.03, "client": "testclient"}


(200,
 {'prediction': 0,
  'label': 'no_heart_disease',
  'confidence': 0.702,
  'risk_probability': 0.298,
  'risk_band': 'low',
  'model_version': 'logistic_regression-1.0.0',
  'request_id': '9d4df3a25a06'})